# NK DSGE Model — Blanchard-Kahn Solution

**Model equations (all in log-deviations from steady state)**

IS curve:  
$$x_t = E_t x_{t+1} - \sigma(i_t - E_t\pi_{t+1} - r^n_t)$$

NKPC:  
$$\pi_t = \beta E_t\pi_{t+1} + \kappa x_t + u_t$$

Taylor rule:  
$$i_t = \rho_i i_{t-1} + (1-\rho_i)[r^* + \phi_\pi \pi_t + \phi_y x_t] + \varepsilon^m_t$$

Shock processes:
$$z_t = \rho_z z_{t-1} + \varepsilon_t, \quad z \in \{a, d, u\}$$

**State vector:** $s_t = [x_t, \pi_t, i_t, z_t]'$  
**Shocks:** technology ($a_t$), demand ($d_t$), cost-push ($u_t$)

The BK solution gives the VAR(1) representation:
$$s_t = A \cdot s_{t-1} + B \cdot \varepsilon_t$$

We export $A$, $B$, and the efficient/natural-rate loadings to `model_matrices.json`.

In [ ]:
import numpy as np
from scipy.linalg import ordqz, solve
import json
import warnings
warnings.filterwarnings('ignore')

## 1. Calibration
Adjust parameters here. Everything below re-solves automatically.

In [ ]:
# --- Structural parameters ---
beta  = 0.99    # household discount factor
sigma = 1.0     # inverse EIS (risk aversion)
kappa = 0.15    # slope of NKPC (function of price stickiness)
eta   = 1.0     # inverse Frisch elasticity of labour supply

# --- Taylor rule ---
phi_pi  = 1.50  # inflation response (must be > 1 for BK)
phi_y   = 0.50  # output gap response
rho_i   = 0.70  # interest rate smoothing

# --- Steady state ---
r_star  = (1/beta - 1)  # net real rate ≈ 0.01
pi_star = 0.0           # inflation target (already in deviations)

print(f"Steady-state real rate r*: {r_star:.4f} ({r_star*400:.2f}% annualised)")

## 2. Blanchard-Kahn Solution

We write the model as $\Gamma_0 s_t = \Gamma_1 s_{t-1} + \Psi \varepsilon_t + \Pi \eta_t$  
where $\eta_t = E_t[s_{t+1}] - s_{t+1}$ are forecast errors.  
BK theorem: if #unstable eigenvalues = #forward-looking variables, a unique stable solution exists.

In [ ]:
def solve_nk(phi_pi=1.5, phi_y=0.5, rho_i=0.70,
             beta=0.99, sigma=1.0, kappa=0.15, eta=1.0,
             rho_shock=0.70, shock_type='tech'):
    """
    Solve 3-equation NK model via Blanchard-Kahn (generalised Schur decomposition).

    State vector: [x_t, pi_t, i_t, z_t]
      x   = output gap
      pi  = inflation deviation
      i   = nominal rate deviation
      z   = exogenous shock (AR1)

    Returns: A (4x4 transition), B (4x1 impact), y_eff_loadings, r_nat_loadings
    """

    # Number of variables
    # Forward-looking: x_t, pi_t  (jump variables)
    # Predetermined:   i_{t-1} (via smoothing), z_t (shock)

    # --- Build system matrices ---
    # Order: [x, pi, i, z]
    # Equations:
    #  (1) IS:      x = E[x+] - sigma*(i - E[pi+]) + loading*z
    #  (2) NKPC:    pi = beta*E[pi+] + kappa*x + loading*z
    #  (3) Taylor:  i = rho_i*i- + (1-rho_i)*(phi_pi*pi + phi_y*x) + loading*z
    #  (4) Shock:   z = rho_shock*z-

    # Shock loadings per type
    if shock_type == 'tech':
        # Technology: enters natural rate & efficient output
        # r^n_t = r* + sigma*(1+eta)/(sigma+eta) * rho_a * a_t  (approx)
        alpha_IS   = sigma * (1 + eta) / (sigma + eta) * rho_shock   # IS demand via r^n
        alpha_NKPC = 0.0
        alpha_TR   = 0.0   # TR doesn't directly load on tech
    elif shock_type == 'demand':
        # Demand: enters IS directly
        alpha_IS   = 1.0
        alpha_NKPC = 0.0
        alpha_TR   = 0.0
    elif shock_type == 'costpush':
        # Cost-push: enters NKPC directly
        alpha_IS   = 0.0
        alpha_NKPC = 1.0
        alpha_TR   = 0.0

    # --- Gamma0 * s_t = Gamma1 * s_{t-1} + Psi * eps + Pi * eta ---
    # Variables: [x, pi, i, z, Ex_next, Epi_next]
    # We use the standard companion form

    # Reduced-form solution via method of undetermined coefficients
    # Guess: s_t = P * s_{t-1} + Q * eps_t
    # For the 3-eq NK with one shock AR(1), this has clean closed form

    # Coefficient matrix for [x_t, pi_t] in terms of [x_{t-1}...] doesn't exist
    # since both are jump variables. Use Schur decomposition on the 2x2 system
    # after substituting out i_t using the Taylor rule.

    # Substituting Taylor rule into IS:
    # x = E[x+] - sigma*((1-rho_i)*(phi_pi*pi + phi_y*x) + rho_i*i- - E[pi+]) + alpha_IS*z

    # Full 4-variable system matrix form
    # G0 * [x,pi,i,z]' = G1 * [x-,pi-,i-,z-]' + impact
    # where - denotes t-1 and we use expectations

    # We solve the 2x2 forward-looking block {x, pi} given predetermined {i, z}
    # then reconstruct full transition matrix

    # --- Solve 2x2 RE system ---
    # Let y = [x, pi]', predetermined p = [i, z]'
    # System: A11 E[y+] + A12 y + A13 p = 0 (structural)

    # IS: x - E[x+] + sigma*(1-rho_i)*phi_pi*pi + sigma*(1-rho_i)*phi_y*x
    #       - sigma*E[pi+] + sigma*rho_i*i - alpha_IS*z = 0
    # NKPC: pi - beta*E[pi+] - kappa*x - alpha_NKPC*z = 0

    # Coefficients on E[y+] = [E[x+], E[pi+]]
    A_fwd = np.array([
        [-1.0,  sigma],   # IS: -E[x+] + sigma*E[pi+] ... wait, sign:
        [ 0.0, -beta ]    # NKPC: -beta*E[pi+]
    ])
    # Coefficients on y = [x, pi]
    A_cur = np.array([
        [1 + sigma*(1-rho_i)*phi_y,  sigma*(1-rho_i)*phi_pi],
        [-kappa,                      1.0                    ]
    ])
    # Coefficients on p = [i, z]
    A_pre = np.array([
        [sigma*rho_i,  -alpha_IS ],
        [0.0,          -alpha_NKPC]
    ])

    # Solve: A_fwd * E[y+] = -A_cur * y - A_pre * p
    # => E[y+] = -A_fwd^{-1} A_cur y - A_fwd^{-1} A_pre p
    Afwd_inv = np.linalg.inv(A_fwd)
    M = -Afwd_inv @ A_cur   # [2x2] coefficient on y
    N = -Afwd_inv @ A_pre   # [2x2] coefficient on p

    # System: E[y+] = M * y + N * p
    # BK: eigenvalues of M, both should be outside unit circle for 2 jump vars
    eigvals = np.linalg.eigvals(M)
    n_explosive = np.sum(np.abs(eigvals) > 1)
    print(f"  Eigenvalues of forward-looking block: {eigvals}")
    print(f"  Explosive roots: {n_explosive} (need 2 for unique solution)")

    if n_explosive != 2:
        print("  WARNING: BK condition not satisfied! Check Taylor principle (phi_pi > 1).")

    # Schur decomposition: M = Z T Z'
    # Order stable roots first
    from scipy.linalg import schur, rsf2csf
    T_schur, Z_schur = schur(M, output='real')

    # Since both roots explosive, stable solution requires the jump to
    # eliminate explosive paths: y_t = -M^{-1} * N * p_t (impact effect)
    # Full solution: y_t = C * p_t where C = -(M - I)^{-1} * ... 
    # Use the method of undetermined coefficients: y_t = F * p_t
    # Substituting: M*F = F*diag(rho_i, rho_shock) + N  ... solve Sylvester eq

    from scipy.linalg import solve_sylvester
    P_pre = np.diag([rho_i, rho_shock])  # transition of predetermined

    # Sylvester: M*F - F*P_pre = N  =>  solve_sylvester(M, -P_pre, N)
    # scipy: AXB + X = Q  ->  not standard, use direct solve
    # MF - FP = N: vec(F) solution via Kronecker
    n = 2
    A_kron = np.kron(P_pre.T, np.eye(n)) - np.kron(np.eye(n), M)
    F_vec = np.linalg.solve(A_kron, N.flatten(order='F'))
    F = F_vec.reshape(n, n, order='F')  # [2x2]: y = F * p

    # Impact of shock on jump variables
    # p_t = P_pre * p_{t-1} + [0, 1]' * eps  (shock enters z only)
    # y_t = F * p_t  =>  dy/deps = F * [0,1]' = F[:,1]

    # --- Build full 4x4 transition matrix A and impact B ---
    # State: [x, pi, i, z]
    # x_t   = F[0,0]*i_{t-1} + F[0,1]*z_t   but z_t = rho*z_{t-1} + eps
    # pi_t  = F[1,0]*i_{t-1} + F[1,1]*z_t
    # i_t   = (1-rho_i)*(phi_pi*pi_t + phi_y*x_t) + rho_i*i_{t-1} + alpha_TR*eps
    # z_t   = rho_shock * z_{t-1} + eps

    # Express everything in terms of [x_{t-1}, pi_{t-1}, i_{t-1}, z_{t-1}]
    # Since x, pi are jump vars with no lag, transition is through z and i only

    f00, f01 = F[0, 0], F[0, 1]  # x = f00*i + f01*z
    f10, f11 = F[1, 0], F[1, 1]  # pi = f10*i + f11*z

    # i_t = (1-rho_i)*(phi_pi*(f10*i_{t-1}+f11*z_t) + phi_y*(f00*i_{t-1}+f01*z_t)) + rho_i*i_{t-1}
    # z_t = rho_shock*z_{t-1}
    # Substitute z_t = rho_shock*z_{t-1} + eps:

    c_i_from_i = rho_i + (1-rho_i)*(phi_pi*f10 + phi_y*f00)   # coeff on i_{t-1} in i_t eq
    c_i_from_z = (1-rho_i)*(phi_pi*f11 + phi_y*f01)*rho_shock  # coeff on z_{t-1}

    A = np.zeros((4, 4))
    # x row: [x-, pi-, i-, z-]
    A[0, 2] = f00                        # from i_{t-1}
    A[0, 3] = f01 * rho_shock            # from z_{t-1} via z_t=rho*z-
    # pi row
    A[1, 2] = f10
    A[1, 3] = f11 * rho_shock
    # i row
    A[2, 2] = c_i_from_i
    A[2, 3] = c_i_from_z
    # z row
    A[3, 3] = rho_shock

    # Impact vector B: effect of unit shock eps on [x, pi, i, z]
    # z_t = rho*z_{t-1} + eps  => dz/deps = 1
    # x_t = f01*z_t => dx/deps = f01
    # pi_t = f11*z_t => dpi/deps = f11
    # i_t = (1-rho_i)*(phi_pi*f11 + phi_y*f01) + alpha_TR
    di_deps = (1-rho_i)*(phi_pi*f11 + phi_y*f01) + alpha_TR

    B = np.array([[f01], [f11], [di_deps], [1.0]])

    # --- Efficient output and natural rate ---
    # Efficient output y* (flexible price, zero markup):
    # y*_t = (1+eta)/(sigma+eta) * a_t  (for tech shock only)
    # For demand: y* doesn't move; for cost-push: y* doesn't move
    if shock_type == 'tech':
        y_eff_loading = (1 + eta) / (sigma + eta)   # per unit of z_t
    else:
        y_eff_loading = 0.0

    # Natural rate r^n_t = r* + sigma*(1+eta)/(sigma+eta)*rho_a*a_t (tech)
    #                     = r* + rho_d * d_t / sigma (demand)
    if shock_type == 'tech':
        r_nat_loading = sigma * (1 + eta) / (sigma + eta) * rho_shock
    elif shock_type == 'demand':
        r_nat_loading = rho_shock / sigma
    else:
        r_nat_loading = 0.0

    return A, B, y_eff_loading, r_nat_loading, eigvals


# Test with default calibration
print("Solving for each shock type...")
for st in ['tech', 'demand', 'costpush']:
    print(f"\n--- {st.upper()} SHOCK ---")
    A, B, y_eff, r_nat, eigs = solve_nk(
        phi_pi=phi_pi, phi_y=phi_y, rho_i=rho_i,
        beta=beta, sigma=sigma, kappa=kappa, eta=eta,
        rho_shock=0.70, shock_type=st
    )
    print(f"  y_eff loading: {y_eff:.4f}")
    print(f"  r_nat loading: {r_nat:.4f}")

## 3. Export matrices to JSON

We solve for each shock type at the **baseline** persistence (ρ=0.70).  
The website will rescale shock size via the impact vector B — persistence changes recompute A and B on the fly using the same structural parameters.

We export:
- `params`: structural calibration (so the site can re-solve if user tweaks Taylor rule)
- Per shock: `A`, `B`, `y_eff_loading`, `r_nat_loading`

In [ ]:
def solve_all_shocks(phi_pi, phi_y, rho_i, beta, sigma, kappa, eta, rho_shock):
    """Solve for all three shock types and return combined dict."""
    result = {}
    for st in ['tech', 'demand', 'costpush']:
        A, B, y_eff, r_nat, _ = solve_nk(
            phi_pi=phi_pi, phi_y=phi_y, rho_i=rho_i,
            beta=beta, sigma=sigma, kappa=kappa, eta=eta,
            rho_shock=rho_shock, shock_type=st
        )
        result[st] = {
            'A': A.tolist(),
            'B': B.flatten().tolist(),
            'y_eff_loading': float(y_eff),
            'r_nat_loading': float(r_nat)
        }
    return result


# Default rho used for baseline export
# (HTML recomputes when user drags persistence slider — see notes below)
rho_default = 0.70

matrices = solve_all_shocks(phi_pi, phi_y, rho_i, beta, sigma, kappa, eta, rho_default)

output = {
    'params': {
        'beta':     beta,
        'sigma':    sigma,
        'kappa':    kappa,
        'eta':      eta,
        'phi_pi':   phi_pi,
        'phi_y':    phi_y,
        'rho_i':    rho_i,
        'r_star':   r_star,
        'rho_default': rho_default
    },
    'shocks': matrices
}

with open('model_matrices.json', 'w') as f:
    json.dump(output, f, indent=2)

print("Exported to model_matrices.json")
print(json.dumps(output['params'], indent=2))

## 4. Compute impulse responses (verification)

Check that the IRFs look economically sensible before handing off to the website.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

T = 40
shock_size = 1.0  # one standard deviation

fig = plt.figure(figsize=(15, 10))
fig.suptitle('Impulse Response Functions — NK DSGE (Blanchard-Kahn solution)', fontsize=13, y=0.98)
gs = gridspec.GridSpec(3, 4, figure=fig, hspace=0.5, wspace=0.35)

shock_labels = {'tech': 'Technology shock', 'demand': 'Demand shock', 'costpush': 'Cost-push shock'}
var_labels   = ['Output gap (x)', 'Inflation (π)', 'Nominal rate (i)', 'Shock (z)']
colors       = ['#378ADD', '#1D9E75', '#D85A30', '#888780']

for row, st in enumerate(['tech', 'demand', 'costpush']):
    A, B, y_eff, r_nat, _ = solve_nk(
        phi_pi=phi_pi, phi_y=phi_y, rho_i=rho_i,
        beta=beta, sigma=sigma, kappa=kappa, eta=eta,
        rho_shock=rho_default, shock_type=st
    )

    # Simulate IRF
    states = np.zeros((T, 4))
    states[0] = B.flatten() * shock_size
    for t in range(1, T):
        states[t] = A @ states[t-1]

    # Efficient output path
    z_path = np.array([shock_size * rho_default**t for t in range(T)])
    y_eff_path = y_eff * z_path

    quarters = np.arange(T)
    for col in range(4):
        ax = fig.add_subplot(gs[row, col])
        ax.axhline(0, color='#ccc', linewidth=0.8)
        ax.plot(quarters, states[:, col], color=colors[col], linewidth=1.8)
        if col == 0 and y_eff != 0:
            ax.plot(quarters, y_eff_path, '--', color='#1D9E75', linewidth=1.2, label='y*')
            ax.legend(fontsize=8)
        if row == 0:
            ax.set_title(var_labels[col], fontsize=10)
        if col == 0:
            ax.set_ylabel(shock_labels[st], fontsize=9)
        ax.tick_params(labelsize=8)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

plt.savefig('irf_verification.png', dpi=150, bbox_inches='tight')
plt.show()
print("IRF plot saved to irf_verification.png")

## 5. HP filter utility (for reference — also reimplemented in JS)

The website replicates this in JavaScript for the trend display. This cell lets you verify the Python and JS versions give the same output.

In [ ]:
def hp_filter(y, lam=1600):
    """
    Hodrick-Prescott filter.
    Returns (trend, cycle) tuple.
    lam=1600 is standard for quarterly data.
    """
    n = len(y)
    y = np.array(y)

    # Build second-difference matrix
    D = np.zeros((n-2, n))
    for i in range(n-2):
        D[i, i]   =  1
        D[i, i+1] = -2
        D[i, i+2] =  1

    # Solve: (I + lam * D'D) * trend = y
    trend = np.linalg.solve(np.eye(n) + lam * D.T @ D, y)
    cycle = y - trend
    return trend, cycle


# Quick test
A_test, B_test, y_eff_test, _, _ = solve_nk(
    phi_pi=phi_pi, phi_y=phi_y, rho_i=rho_i,
    beta=beta, sigma=sigma, kappa=kappa, eta=eta,
    rho_shock=rho_default, shock_type='tech'
)
states_test = np.zeros((T, 4))
states_test[0] = B_test.flatten()
for t in range(1, T):
    states_test[t] = A_test @ states_test[t-1]

trend, cycle = hp_filter(states_test[:, 0])
print("HP filter check — first 5 trend values:")
print(np.round(trend[:5], 4))

## Notes for extending this model

**Towards SW-style model:**
- Add capital accumulation: introduces investment Euler equation and capital rental rate
- Add habit formation in consumption: modifies IS curve with lagged output
- Add wage stickiness: second NKPC for wage inflation
- Add price indexation: modifies NKPC to include lagged inflation
- The BK approach generalises directly — state vector grows but solution method is identical

**VAR representation:**
Any linearised DSGE has a VAR(1) representation S_t = A*S_{t-1} + B*eps_t  
For larger models, extract this from Dynare's `oo_.dr` structure or use `gensys` (Sims, 2002).

**Re-exporting when Taylor parameters change:**
Run `solve_all_shocks(...)` with new phi_pi/phi_y and overwrite model_matrices.json.  
The website picks up the new file on reload — no other changes needed.